In [1]:
# Get raw data
with open('input/16.txt', 'r') as f:
    rawinput = f.read().strip()

In [2]:
import numpy as np

In [3]:
# Part 1
flow_rates = {j[1]: z
              for i in rawinput.split('\n') 
              for j in [i.split(';')[0].split()]
              if (z:=int(j[4].split('=')[1]))}
tunnels = {i.split()[1]: [k.replace(',','') for k in j] 
           for i in rawinput.split('\n') 
           for j in [i.split(';')[1].split()[4:]]}

loc = np.array(['AA'])
open_valves = np.empty((1,0), dtype='<U2')
tot_pressure = np.array([0])
hist = {'AA-'}
n_minutes = 30

for minute in range(n_minutes-1,-1,-1):
    rpt = (np.vectorize(lambda x: len(tunnels[x]))(loc) 
           + (np.isin(loc, [*flow_rates]) & np.all(np.expand_dims(loc,1) != open_valves, axis=1)))
    loc, ov_add = [np.array(l, dtype='<U2') 
                   for l in zip(*[[k, j if j==k else '__'] 
                                  for i,j in enumerate(loc)
                                  for k in [*tunnels[j], 
                                            *([j] if j in flow_rates and j not in open_valves[i] else [])]])]
    open_valves = np.sort(np.column_stack((np.repeat(open_valves, rpt, axis=0), 
                                           ov_add)), axis=1)
    if np.any(z:=np.all(open_valves=='__', axis=0)):
        open_valves = open_valves[:, :np.argmax(z)]
    tot_pressure = np.repeat(tot_pressure, rpt) + np.vectorize(lambda x: flow_rates.get(x,0))(ov_add) * minute
    
    ovsumm = (np.where(open_valves=='__', '', open_valves)
                .reshape(-1).view(f'<U{max(2,z*2)}') 
              if (z:=open_valves.shape[1]) 
              else np.full_like(loc, ''))
    hist_id = np.char.add(np.char.add(loc, ['-']*loc.shape[0]), ovsumm)
    
    filt = np.vectorize(lambda x: x not in hist)(hist_id)
    loc, open_valves, tot_pressure = [i[filt] for i in [loc, open_valves, tot_pressure]]
    hist.update(hist_id.tolist())
    
max(tot_pressure).item()

1737

In [4]:
# Part 2
loc = np.array(['AA'])
open_valves = np.empty((1,0), dtype='<U2')
tot_pressure = np.array([0])
hist = {'AA-'}
n_minutes = 26

for minute in range(n_minutes-1,-1,-1):
    rpt = (np.vectorize(lambda x: len(tunnels[x]))(loc) 
           + (np.isin(loc, [*flow_rates]) & np.all(np.expand_dims(loc,1) != open_valves, axis=1)))
    loc, ov_add = [np.array(l, dtype='<U2') 
                   for l in zip(*[[k, j if j==k else '__'] 
                                  for i,j in enumerate(loc)
                                  for k in [*tunnels[j], 
                                            *([j] if j in flow_rates and j not in open_valves[i] else [])]])]
    open_valves = np.sort(np.column_stack((np.repeat(open_valves, rpt, axis=0), 
                                           ov_add)), axis=1)
    if np.any(z:=np.all(open_valves=='__', axis=0)):
        open_valves = open_valves[:, :np.argmax(z)]
    tot_pressure = np.repeat(tot_pressure, rpt) + np.vectorize(lambda x: flow_rates.get(x,0))(ov_add) * minute
    
    ovsumm = (np.where(open_valves=='__', '', open_valves)
                .reshape(-1).view(f'<U{max(2,z*2)}') 
              if (z:=open_valves.shape[1]) 
              else np.full_like(loc, ''))
    hist_id = np.char.add(np.char.add(loc, ['-']*loc.shape[0]), ovsumm)
    
    filt = np.vectorize(lambda x: x not in hist)(hist_id)
    loc, open_valves, tot_pressure = [i[filt] for i in [loc, open_valves, tot_pressure]]
    hist.update(hist_id.tolist())
    
max_by_open_valves = dict(zip(*[*zip(*sorted([[j[3:], tot_pressure[i].item()] 
                                              for i,j in enumerate(hist_id[filt])]))]))
max([max_by_open_valves[i]+max_by_open_valves[j]
     for i in max_by_open_valves
     for j in max_by_open_valves
     if j>i 
     and len(z:=[k[l:l+2] for k in [i,j] for l in range(0,len(k),2)]) == len({*z})])

2216